# Insider Threat Detection Model Training
## Using CERT Dataset and LSTM Autoencoder

This notebook trains an insider threat detection model that identifies:
- 🔴 **Brute Force Attacks** - Multiple failed logins
- 🔴 **Location Hopping** - Rapid IP changes
- 🔴 **Off-Hours Privileged Access** - Admin actions at unusual times
- 🔴 **Data Exfiltration** - Large data transfers
- 🔴 **Privilege Escalation** - Sudden privilege increases
- 🔴 **Unusual Resource Access** - Access to new sensitive resources

**Dataset:** CERT Insider Threat Test Dataset from CMU KiltHub  
**Model:** LSTM Autoencoder (7 timesteps × 20 features)


In [ ]:
# Step 1: Install Required Libraries
print("📦 Installing required packages...")

!pip install -q tensorflow pandas numpy scikit-learn matplotlib seaborn requests tqdm

print("✅ Installation complete!")


In [ ]:
# Step 2: Upload Feature Extractor and Dataset
print("📤 Please upload the following files:")
print("   1. colab_feature_extractor.py")
print("   2. cert_dataset.tar.bz2 (from KiltHub)")

from google.colab import files
import os

# Upload feature extractor
print("\n📄 Upload colab_feature_extractor.py:")
uploaded = files.upload()

# Verify feature extractor was uploaded
if 'colab_feature_extractor.py' in uploaded:
    print("✅ Feature extractor uploaded successfully!")
else:
    print("⚠️ Warning: colab_feature_extractor.py not found. Please upload it.")


In [ ]:
# Step 3: Upload CERT Dataset
print("📂 Upload CERT dataset (cert_dataset.tar.bz2):")
print("   Download from: https://kilthub.cmu.edu/articles/dataset/Insider_Threat_Test_Dataset/12841247/1")
print("   Direct link: https://kilthub.cmu.edu/ndownloader/articles/12841247/versions/1")

uploaded_dataset = files.upload()

# Check if dataset was uploaded
if any('cert' in name.lower() and '.tar' in name.lower() for name in uploaded_dataset.keys()):
    dataset_file = [name for name in uploaded_dataset.keys() if 'cert' in name.lower() and '.tar' in name.lower()][0]
    print(f"✅ Dataset uploaded: {dataset_file}")
else:
    print("⚠️ Warning: CERT dataset not found. Please upload cert_dataset.tar.bz2")


In [ ]:
# Step 4: Extract Dataset
import tarfile
import os

# Find the dataset file
dataset_files = [f for f in os.listdir('.') if 'cert' in f.lower() and '.tar' in f.lower()]
if not dataset_files:
    print("❌ No CERT dataset found. Please upload it first.")
else:
    dataset_file = dataset_files[0]
    print(f"📦 Extracting {dataset_file}...")
    
    # Create extraction directory
    os.makedirs('extracted', exist_ok=True)
    
    # Extract
    with tarfile.open(dataset_file, 'r:bz2') as tar:
        tar.extractall('extracted')
    
    print("✅ Dataset extracted successfully!")
    print(f"📁 Extracted to: extracted/")
    
    # List extracted files
    print("\n📋 Extracted files:")
    !find extracted -name "*.csv" -o -name "*.txt" | head -20


In [ ]:
# Step 5: Import Feature Extractor
print("🔧 Loading feature extractor...")

try:
    from colab_feature_extractor import InsiderThreatFeatureExtractor
    print("✅ Feature extractor loaded successfully!")
except ImportError as e:
    print(f"❌ Error loading feature extractor: {e}")
    print("Please make sure colab_feature_extractor.py is uploaded in the correct cell.")


In [ ]:
# Step 6: Load and Explore CERT Dataset
import pandas as pd
import numpy as np
from pathlib import Path
import glob

print("📊 Loading CERT dataset...")

# Find all CSV files in extracted directory
csv_files = glob.glob('extracted/**/*.csv', recursive=True)
csv_files += glob.glob('extracted/**/*.txt', recursive=True)

print(f"Found {len(csv_files)} files")

# Try to find logon data (common in CERT dataset)
logon_files = [f for f in csv_files if 'logon' in f.lower() or 'login' in f.lower()]

if logon_files:
    print(f"\n📂 Loading logon data from: {logon_files[0]}")
    try:
        # CERT dataset often uses tab-separated or space-separated values
        df = pd.read_csv(logon_files[0], sep='\t', low_memory=False, on_bad_lines='skip')
        if len(df.columns) == 1:
            df = pd.read_csv(logon_files[0], sep=',', low_memory=False, on_bad_lines='skip')
    except:
        df = pd.read_csv(logon_files[0], low_memory=False, on_bad_lines='skip')
    
    print(f"✅ Loaded {len(df)} rows, {len(df.columns)} columns")
    print(f"\nColumn names: {list(df.columns)}")
    print(f"\nFirst few rows:")
    print(df.head())
    print(f"\nData types:")
    print(df.dtypes)
    print(f"\nBasic statistics:")
    print(df.describe())
else:
    print("⚠️ No logon files found. Loading first CSV file...")
    if csv_files:
        df = pd.read_csv(csv_files[0], low_memory=False, on_bad_lines='skip')
        print(f"✅ Loaded {len(df)} rows from {csv_files[0]}")
        print(f"\nColumn names: {list(df.columns)}")
        print(df.head())
    else:
        print("❌ No CSV files found. Please check dataset extraction.")
        df = None


In [ ]:
# Step 7: Extract Features from Dataset
if df is not None and len(df) > 0:
    print("🔧 Extracting features from dataset...")
    
    # Initialize feature extractor
    extractor = InsiderThreatFeatureExtractor()
    
    # Extract features (this may take a while for large datasets)
    # Use a sample if dataset is too large
    sample_size = min(50000, len(df))  # Process up to 50k rows for training
    if len(df) > sample_size:
        print(f"📊 Using sample of {sample_size:,} rows (dataset has {len(df):,} rows)")
        df_sample = df.sample(n=sample_size, random_state=42).sort_index()
    else:
        df_sample = df
    
    # Extract features
    feature_df = extractor.extract_features(df_sample.copy())
    
    print(f"\n✅ Feature extraction complete!")
    print(f"📊 Extracted features from {len(feature_df)} events")
    print(f"📋 Feature vectors shape: {len(feature_df['features'].iloc[0])} features per event")
    
    # Display sample
    print("\n📋 Sample feature vector:")
    print(feature_df.head())
else:
    print("❌ No data to process. Please check previous steps.")


In [ ]:
# Step 8: Prepare Data for LSTM Autoencoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("🔄 Preparing data for LSTM Autoencoder...")

if 'feature_df' in locals() and len(feature_df) > 0:
    # Extract feature vectors
    X_raw = np.array([f for f in feature_df['features']])
    
    print(f"📊 Raw features shape: {X_raw.shape}")
    print(f"   - Samples: {X_raw.shape[0]}")
    print(f"   - Features per sample: {X_raw.shape[1]}")
    
    # Normalize features
    print("\n📐 Normalizing features...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    
    # Create sequences (7 timesteps)
    timesteps = 7
    sequences = []
    
    print(f"\n🔗 Creating sequences with {timesteps} timesteps...")
    for i in range(timesteps, len(X_scaled)):
        sequences.append(X_scaled[i-timesteps:i])
    
    X = np.array(sequences)
    print(f"✅ Sequences created: {X.shape}")
    print(f"   - Samples: {X.shape[0]}")
    print(f"   - Timesteps: {X.shape[1]}")
    print(f"   - Features per timestep: {X.shape[2]}")
    
    # Split into train/test
    X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
    
    print(f"\n📊 Train set: {X_train.shape[0]} sequences")
    print(f"📊 Test set: {X_test.shape[0]} sequences")
    
    print("\n✅ Data preparation complete!")
else:
    print("❌ No features available. Please run feature extraction first.")


In [ ]:
# Step 9: Build LSTM Autoencoder Model
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("🏗️ Building LSTM Autoencoder model...")

# Model parameters
timesteps = 7
n_features = 20
encoding_dim = 32

print(f"📐 Model architecture:")
print(f"   - Input: ({timesteps}, {n_features})")
print(f"   - Encoding dimension: {encoding_dim}")

# Build model
input_layer = layers.Input(shape=(timesteps, n_features))

# Encoder
encoder = layers.LSTM(128, return_sequences=True)(input_layer)
encoder = layers.Dropout(0.2)(encoder)
encoder = layers.LSTM(64, return_sequences=False)(encoder)

# Bottleneck
encoded = layers.Dense(encoding_dim, activation='relu')(encoder)

# Decoder
decoder = layers.RepeatVector(timesteps)(encoded)
decoder = layers.LSTM(64, return_sequences=True)(decoder)
decoder = layers.Dropout(0.2)(decoder)
decoder = layers.LSTM(128, return_sequences=True)(decoder)

# Output
decoder = layers.TimeDistributed(
    layers.Dense(n_features, activation='linear')
)(decoder)

# Create model
model = keras.Model(input_layer, decoder)

# Compile
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# Display model summary
print("\n📋 Model Summary:")
model.summary()

print("\n✅ Model built successfully!")


In [ ]:
# Step 10: Train the Model
print("🚀 Starting model training...")
print("⏱️ This may take 1-2 hours depending on dataset size and GPU availability")

# Training parameters
batch_size = 32
epochs = 50

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_insider_threat_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# Train
if 'X_train' in locals() and X_train.shape[0] > 0:
    history = model.fit(
        X_train, X_train,  # Autoencoder: input = output
        validation_data=(X_test, X_test),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\n✅ Training complete!")
    print(f"📁 Best model saved to: best_insider_threat_model.h5")
else:
    print("❌ No training data available. Please prepare data first.")


In [ ]:
# Step 11: Evaluate Model Performance
import matplotlib.pyplot as plt

print("📊 Evaluating model performance...")

if 'model' in locals() and 'X_test' in locals():
    # Evaluate on test set
    test_loss, test_mae = model.evaluate(X_test, X_test, verbose=0)
    print(f"\n📈 Test Loss (MSE): {test_loss:.6f}")
    print(f"📈 Test MAE: {test_mae:.6f}")
    
    # Calculate reconstruction errors
    print("\n🔍 Calculating reconstruction errors...")
    predictions = model.predict(X_test, verbose=0)
    reconstruction_errors = np.mean(np.square(X_test - predictions), axis=(1, 2))
    
    print(f"\n📊 Reconstruction Error Statistics:")
    print(f"   - Mean: {np.mean(reconstruction_errors):.6f}")
    print(f"   - Std: {np.std(reconstruction_errors):.6f}")
    print(f"   - Min: {np.min(reconstruction_errors):.6f}")
    print(f"   - Max: {np.max(reconstruction_errors):.6f}")
    print(f"   - 95th percentile: {np.percentile(reconstruction_errors, 95):.6f}")
    
    # Calculate threshold (mean + 2*std)
    threshold = np.mean(reconstruction_errors) + 2 * np.std(reconstruction_errors)
    print(f"\n🎯 Recommended Anomaly Threshold: {threshold:.6f}")
    print(f"   (Mean + 2*Std)")
    
    # Plot training history
    if 'history' in locals():
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        plt.plot(history.history['loss'], label='Train Loss')
        plt.plot(history.history['val_loss'], label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss (MSE)')
        plt.title('Model Loss')
        plt.legend()
        plt.grid(True)
        
        plt.subplot(1, 2, 2)
        plt.plot(history.history['mae'], label='Train MAE')
        plt.plot(history.history['val_mae'], label='Val MAE')
        plt.xlabel('Epoch')
        plt.ylabel('MAE')
        plt.title('Model MAE')
        plt.legend()
        plt.grid(True)
        
        plt.tight_layout()
        plt.show()
    
    # Plot reconstruction error distribution
    plt.figure(figsize=(10, 6))
    plt.hist(reconstruction_errors, bins=50, alpha=0.7, edgecolor='black')
    plt.axvline(threshold, color='r', linestyle='--', label=f'Threshold: {threshold:.6f}')
    plt.xlabel('Reconstruction Error (MSE)')
    plt.ylabel('Frequency')
    plt.title('Reconstruction Error Distribution')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("\n✅ Evaluation complete!")
else:
    print("❌ Model or test data not available.")


In [ ]:
# Step 12: Save Model and Metadata
import json
import pickle

print("💾 Saving model and metadata...")

if 'model' in locals():
    # Load best model if it exists (from training checkpoint)
    if os.path.exists('best_insider_threat_model.h5'):
        print("📂 Loading best model from training checkpoint...")
        model.load_weights('best_insider_threat_model.h5')
    
    # Save final model
    model_filename = 'insider_threat_model_cert.h5'
    model.save(model_filename)
    print(f"✅ Model saved: {model_filename}")
    
    # Save scaler
    if 'scaler' in locals():
        scaler_filename = 'feature_scaler.pkl'
        with open(scaler_filename, 'wb') as f:
            pickle.dump(scaler, f)
        print(f"✅ Scaler saved: {scaler_filename}")
    
    # Save metadata
    metadata = {
        'model_type': 'LSTM Autoencoder',
        'timesteps': 7,
        'n_features': 20,
        'encoding_dim': 32,
        'architecture': {
            'encoder': ['LSTM(128)', 'Dropout(0.2)', 'LSTM(64)'],
            'bottleneck': 'Dense(32, relu)',
            'decoder': ['RepeatVector(7)', 'LSTM(64)', 'Dropout(0.2)', 'LSTM(128)', 'TimeDistributed(Dense(20))']
        },
        'training': {
            'batch_size': 32,
            'optimizer': 'adam',
            'loss': 'mse'
        }
    }
    
    # Add threshold if available
    if 'threshold' in locals():
        metadata['anomaly_threshold'] = float(threshold)
    
    # Add evaluation metrics if available
    if 'test_loss' in locals():
        metadata['evaluation'] = {
            'test_loss': float(test_loss),
            'test_mae': float(test_mae) if 'test_mae' in locals() else None
        }
    
    metadata_filename = 'model_metadata.json'
    with open(metadata_filename, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✅ Metadata saved: {metadata_filename}")
    
    print("\n📦 Files ready for download:")
    print(f"   1. {model_filename}")
    if 'scaler' in locals():
        print(f"   2. {scaler_filename}")
    print(f"   3. {metadata_filename}")
    
    print("\n✅ Save complete!")
else:
    print("❌ Model not available to save.")


In [ ]:
# Step 13: Download Model Files
print("📥 Download your trained model files:")

from google.colab import files

# Download model
if os.path.exists('insider_threat_model_cert.h5'):
    files.download('insider_threat_model_cert.h5')
    print("✅ Model downloaded")

# Download scaler
if os.path.exists('feature_scaler.pkl'):
    files.download('feature_scaler.pkl')
    print("✅ Scaler downloaded")

# Download metadata
if os.path.exists('model_metadata.json'):
    files.download('model_metadata.json')
    print("✅ Metadata downloaded")

print("\n🎉 All files downloaded! You can now integrate the model into your S-UEBA system.")


## Next Steps: Integration with S-UEBA System

After downloading the model files:

1. **Upload to S-UEBA System:**
   - Go to the Models page in your application
   - Click "Upload Model"
   - Select `insider_threat_model_cert.h5`
   - Set model type: `insider_threat`
   - Set framework: `tensorflow`

2. **Configure Model:**
   - Use the threshold value from `model_metadata.json`
   - Set input shape: `(7, 20)` (7 timesteps, 20 features)
   - Enable the model for anomaly detection

3. **Test the Model:**
   - Upload test CSV data with log events
   - Verify anomaly detection works correctly
   - Check that detection patterns match expectations

## Model Capabilities

✅ **Brute Force Attacks** - Detects multiple failed logins in short time  
✅ **Location Hopping** - Identifies rapid IP address changes  
✅ **Off-Hours Privileged Access** - Flags admin actions outside business hours  
✅ **Data Exfiltration** - Detects unusual large data transfers  
✅ **Privilege Escalation** - Identifies sudden privilege increases  
✅ **Unusual Resource Access** - Flags access to new sensitive resources  

---

**Training Complete! 🎉**
